In [1]:
%load_ext bigquery_magics

import bigquery_magics
bigquery_magics.context.project = "k-move1-kyungpil"

## 1. STRUCTデータ型

`STRUCT`は、複数の項目を一つの列にまとめて保存するデータ型である。  
Pythonの`dict`型に似ている。

In [ ]:
%%bigquery

WITH tmp AS (
    SELECT
        STRUCT(
            'キョンピル' AS name,
            27 AS age,
            'ソウル' AS city
        ) AS person,
        'KOREA' AS country

    UNION ALL

    SELECT
        STRUCT(
            'ハウル' AS name,
            25 AS age,
            '京都' AS city
        ) AS person,
        'JAPAN' AS country
)

SELECT
    person,
    country
FROM tmp;

-- `person`列の中に`name`・`age`・`city`がまとめて保存される。
-- `country`は`person`とは別の一般列である。

| 行 | person.name | person.age | person.city | country |
|---:|---|---:|---|---|
| 1 | キョンピル | 27 | ソウル | KOREA |
| 2 | ハウル | 25 | 京都 | JAPAN |

#### 1）ドット演算子でSTRUCT内の値を個別に取得する

In [ ]:
%%bigquery

WITH tmp AS (
    SELECT
        STRUCT(
            'キョンピル' AS name,
            27 AS age,
            'ソウル' AS city
        ) AS person,
        'KOREA' AS country

    UNION ALL

    SELECT
        STRUCT(
            'ハウル' AS name,
            25 AS age,
            '京都' AS city
        ) AS person,
        'JAPAN' AS country
)

SELECT
    person.name,
    person.city,
    country
FROM tmp;

| 行 | person.name | person.city | country |
|---:|---|---|---|
| 1 | キョンピル | ソウル | KOREA |
| 2 | ハウル | 京都 | JAPAN |

#### 2）STRUCTの配列型（ARRAY<STRUCT>）

In [ ]:
%%bigquery

WITH tmp AS (
    SELECT [
        STRUCT('ノートパソコン' AS product, 100 AS amount),
        STRUCT('モニター' AS product, 80 AS amount),
        STRUCT('キーボード' AS product, 30 AS amount)
    ] AS items,
    '2026-08-01' AS order_date
)

SELECT
    items,
    order_date
FROM tmp;

| 行 | items.product | items.amount | order_date |
|---:|---|---:|---|
| 1 | ノートパソコン | 100 | 2026-08-01 |
|  | モニター | 80 |  |
|  | キーボード | 30 |  |

#### - UNNESTを使用して配列を行に展開する例

In [ ]:
%%bigquery

WITH tmp AS (
    SELECT [
        STRUCT('ノートパソコン' AS product, 100 AS amount),
        STRUCT('モニター' AS product, 80 AS amount),
        STRUCT('キーボード' AS product, 30 AS amount)
    ] AS items,
    '2026-01-01' AS order_date
)

SELECT
    tmp.order_date,
    e.product,
    e.amount
FROM tmp, UNNEST(items) AS e;

-- `UNNEST(items)`で配列内の3つのSTRUCTをそれぞれ行に展開する。
-- 元の1行と配列の各要素が結合されるため、`order_date`が繰り返され、結果は3行になる。

| 行 | order_date | product | amount |
|---:|---|---|---:|
| 1 | 2026-01-01 | ノートパソコン | 100 |
| 2 | 2026-01-01 | モニター | 80 |
| 3 | 2026-01-01 | キーボード | 30 |